
# SPARQL - LLM

Este notebook Jupyter, tiene como finalidad una integración y comparativa simple entre una SPARQL y la misma consulta realizada a un LLM.

Primero instalamos las dependencias


In [1]:

!python3.11 -m pip install rdflib


Cargamos los datos RDF

In [1]:

from rdflib import Graph

g = Graph()

g.parse("../Datos/tarea1-multisolve-private.ttl", format="turtle")
g.parse("../Datos/tarea1-chile-geo-open.ttl", format="turtle")

print("Triples loaded:", len(g))


Triples loaded: 157


Consulta SPARQL que tiene como objetivo analizar la distribución geográfica de las órdenes de servicios registradas por MultiSolve, agrupándolas por comuna

In [ ]:

query = """
PREFIX ms:   <http://example.org/multisolve#>
PREFIX geo:  <http://example.org/chile-geo#>

SELECT ?comunaName (COUNT(?so) AS ?totalOrders)
WHERE {
  ?so a ms:ServiceOrder ;
      ms:hasLocation ?loc .
  ?loc ms:locatedInComuna ?comuna .
  ?comuna geo:nombreComuna ?comunaName .
}
GROUP BY ?comunaName
ORDER BY DESC(?totalOrders) ?comunaName

"""

for row in g.query(query):
    print(row)


(rdflib.term.Literal('Santiago', lang='es'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Cerrillos', lang='es'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))


Respuesta simulada desde LLM a la misma pregunta anterior respopndida desde SPARQL, en este caso, ¿cúal es la comuna con más órdenes?

In [ ]:

results = list(g.query(query))

for row in results:
    print(row)

def llm_from_sparql(question, sparql_results):
    if "comuna" in question.lower() and sparql_results:
        top = sparql_results[0]
        comuna = top.comunaName.toPython()
        total = top.totalOrders.toPython()
        return f"La comuna con más órdenes es {comuna}, con {total} órdenes."
    return "No estoy seguro."

question = "¿En qué comuna hay más órdenes de servicio?"
answer = llm_from_sparql(question, results)

print("Pregunta:", question)
print("Respuesta basada en SPARQL:", answer)


(rdflib.term.Literal('Santiago', lang='es'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.Literal('Cerrillos', lang='es'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
Pregunta: ¿En qué comuna hay más órdenes de servicio?
Respuesta basada en SPARQL: La comuna con más órdenes es Santiago, con 2 órdenes.


Como se puede validar, en ambos casos la respuesta es la misma, la comuna con más órdenes es Santiago